# Lesson 01 - Введение в AI-агентов

Добро пожаловать на первый урок курса **AI-агенты для начинающих**!

**AI-агент** — это программа, которая использует большую языковую модель (LLM) в качестве движка рассуждений и может выполнять *действия* в реальном мире — вызывать API, запрашивать базы данных или запускать код — чтобы достичь цели от имени пользователя.

В этом блокноте вы создадите своего первого агента: **Туристический агент**, который рекомендует места для отдыха. По ходу дела вы узнаете, как:

1. Подключиться к службе Azure AI Foundry Agent Service с помощью **Microsoft Agent Framework**.
2. Дать агенту **инструмент** — простую функцию на Python, которую он сможет вызывать.
3. Запустить агента и проверить его ответ.
4. Транслировать ответ агента по токенам.


## Настройка

Перед запуском этой записной книжки убедитесь, что у вас есть:

1. **Проект Azure AI Foundry** с развернутой моделью чата (например, `gpt-4o-mini`).
2. **Вход в систему через Azure CLI** — выполните `az login` в вашем терминале.
3. **Установлены необходимые переменные окружения:**
   - `AZURE_AI_PROJECT_ENDPOINT` — конечная точка вашего проекта Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — имя вашей развернутой модели.

В следующей ячейке устанавливаются необходимые пакеты Python.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Создание вашего первого агента

Агенту нужны две вещи:

- **Инструкции**, которые говорят ему *кто он* и *как себя вести* (системный запрос).
- **Инструменты** — функции Python, украшенные `@tool`, которые агент может вызывать для получения информации или выполнения действий.

Ниже мы определяем простой инструмент, который возвращает список популярных туристических направлений. Агент будет использовать этот инструмент, когда пользователь попросит рекомендации по путешествиям.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Потоковые ответы

Для более интерактивного взаимодействия вы можете **потоково** получать ответ агента. Вместо того чтобы ждать полный ответ, агент выдаёт фрагменты текста по мере их создания. Это особенно полезно в чат-интерфейсах, где вы хотите отображать вывод в режиме реального времени.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

В этом уроке вы узнали, как:

- **Создать провайдера**, который подключается к Azure AI Foundry Agent Service через `AzureAIProjectAgentProvider`.
- **Определить инструмент** с помощью декоратора `@tool`, чтобы агент мог вызывать ваши функции на Python.
- **Запустить агента** с пользовательским сообщением и вывести его ответ.
- **Передавать ответы по потоку** для вывода в реальном времени.

В следующем уроке мы более подробно рассмотрим агентские фреймворки и научимся предоставлять агентам более мощные инструменты и способности многошагового рассуждения.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
